# Laboratorio de Modelos de Clasificación — Ejercicio 3: Breast Cancer

**Plantilla de implementación grupal | Google Colab | Python 3**

Este notebook está **intencionalmente sin código implementado** en las secciones de trabajo. Cada integrante debe completar únicamente las celdas de código que le fueron asignadas y respetar los nombres de variables acordados. Las celdas de texto explican el trabajo requerido, sus dependencias y cómo comprobar que quedó listo.

> **Alcance acordado para el equipo (no es una instrucción adicional atribuida al profesor):** desarrollar el ejercicio original de clasificación de cáncer de mama de principio a fin, con exploración de datos, preprocesamiento, cinco clasificadores individuales y una comparación homogénea. No se incorporan por defecto los cuatro escaladores ni los cinco híbridos del Ejercicio 1 (Santander) porque **no se proporcionó esa indicación para este ejercicio**.

## Organización del trabajo

| Integrante | Secciones / subsecciones asignadas | Entrega principal |
|---|---|---|
| **Marco** *(coordinación)* | 0.1–0.2, 5.1, 5.5, 6.3 y 7.1 | Entorno, contratos, evaluación central, análisis de errores e integración final |
| **Poma** | 1.1–1.3, 5.2 | Carga, validación de datos, variables predictoras y tabla comparativa |
| **Aldana** | 2.1–2.3, 4.1, 5.4, 6.2 | EDA, regresión logística, curvas ROC e interpretación general |
| **Brickzon** | 3.1–3.3, 4.2–4.3 | División estratificada, escalamiento, KNN y SVM |
| **Vera** | 4.4–4.5, 5.3 y 6.1 | Árboles, matrices de confusión e importancia de variables |

**Orden de integración:** Marco (0) → Poma (1) → Aldana (2) → Brickzon (3) → modeladores (4) → responsables de evaluación (5) → cierre (6–7).

Se puede desarrollar código por separado, pero la versión integrada se debe ejecutar **de arriba abajo**, sin depender de variables ocultas de una sesión anterior.

## Descripción del conjunto de datos

El notebook recibido describe el conjunto **Breast Cancer Wisconsin (Diagnostic)**. Las características se calculan sobre imágenes digitalizadas de aspirados con aguja fina (FNA) de masas mamarias y resumen propiedades geométricas y de textura de los núcleos celulares.

- **Problema:** clasificación supervisada binaria, `M` = maligno y `B` = benigno.
- **Observaciones descritas:** 569 (212 malignas y 357 benignas).
- **Predictores descritos:** 30 variables numéricas obtenidas a partir de diez mediciones: radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal; cada medición aparece en forma media, error estándar y valor extremo (worst).
- **Datos de origen:** incluye identificación y diagnóstico. El identificador **no se usa como predictor**. Si aparece una columna vacía de exportación como `Unnamed: 32`, debe descartarse.
- **Valores ausentes:** la descripción original indica que no hay, pero deben comprobarlo en el archivo real.
- **Referencia del notebook original:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).

**Interpretación de clases acordada para este trabajo:** `1 = M (maligno, clase positiva)` y `0 = B (benigno)`. Es obligatorio respetar esta codificación en entrenamiento, métricas, curvas y matrices de confusión. **Precaución:** en `sklearn.datasets.load_breast_cancer()` la codificación de origen es inversa (`0 = malignant`, `1 = benign`); al usar esa alternativa, hay que convertirla.

Este laboratorio es académico: las predicciones de los modelos **no** son diagnósticos clínicos.

## Cómo usar esta plantilla en Google Colab

1. Abran este `.ipynb` en Google Colab y seleccionen el entorno de ejecución **Python 3 (CPU)**; no se requiere GPU.
2. **Opción preferida:** carguen el CSV original de la práctica en la ruta `/content/dataset/data.csv` (equivalente a `dataset/data.csv` si se ejecuta desde `/content`). Pueden crearse la carpeta `dataset` y subir el archivo desde el panel lateral de Colab. Si lo mantienen en Drive, deberán montar Drive y adaptar la ruta en **1.1**.
3. **Alternativa cuando no se dispone del CSV:** en **1.1**, usar `sklearn.datasets.load_breast_cancer(as_frame=True)` y construir un `df_raw` equivalente. **No mezclar fuentes de datos en una misma ejecución.** Documentar cuál se utilizó.
4. Completen las celdas de código vacías que tienen asignadas. No cambien los nombres compartidos definidos en **0.2** sin acordarlo con el equipo.
5. Cuando todas las partes estén listas: **Entorno de ejecución → Reiniciar sesión y ejecutar todo**. Cualquier error en la ejecución desde cero debe corregirse antes de entregar.

**Importante para el trabajo colaborativo:** Colab no fusiona automáticamente dos copias descargadas de un notebook. Coordinen una versión principal y hagan integración secuencial, o utilicen control de versiones para evitar sobrescribir trabajo ajeno.

## 0. Preparación del entorno y acuerdos de integración

### 0.1. Librerías y configuración para Google Colab

**Responsable: Marco.**

**Qué debes hacer**

- Importa las librerías comunes: `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn` y herramientas necesarias de `scikit-learn` para separación, transformación, modelos y métricas.
- Comprueba que el entorno usa Python 3 y que las dependencias están disponibles en Colab. Instala únicamente dependencias ausentes; evita instalar paquetes sin necesidad.
- Fija `RANDOM_STATE = 42` y una semilla global de NumPy para favorecer reproducibilidad. Define preferencias visuales legibles para figuras.

**Resultado esperado / criterio de aceptación**

Una ejecución limpia de esta celda deja listos los imports y `RANDOM_STATE`. No depende de ninguna celda posterior.

In [ ]:
import sys

if sys.version_info.major != 3:
    raise RuntimeError("Este notebook requiere Python 3.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (8, 5), "axes.titlesize": 13,
                     "axes.labelsize": 11, "figure.dpi": 100})
print(f"Entorno listo: Python {sys.version_info.major}.{sys.version_info.minor}; RANDOM_STATE={RANDOM_STATE}")

### 0.2. Variables compartidas y registro de modelos

**Responsable: Marco.**

**Qué debes hacer**

- Declara los contenedores comunes `modelos_entrenados = {}` y `tipo_entrada = {}` antes del entrenamiento.
- Fija y comunica este contrato: `df_raw` = carga original; `df_clean` = columnas limpias; `X` = predictores numéricos; `y` = objetivo codificado con M=1/B=0; `X_train`, `X_test`, `y_train`, `y_test` = particiones; `escalador`, `X_train_scaled`, `X_test_scaled` = transformación ajustada solo en train.
- Cada responsable de un clasificador registrará `modelos_entrenados["Nombre del modelo"] = modelo_ajustado` y `tipo_entrada["Nombre del modelo"] = "escalado"` o `"original"`.
- Los nombres de modelo se acuerdan exactamente así: `Regresión logística`, `KNN`, `SVM`, `Árbol de decisión`, `Random Forest`.

**Resultado esperado / criterio de aceptación**

Existen dos diccionarios vacíos y el equipo conoce nombres y tipos. **No** crear aquí datos artificiales ni entrenar modelos.

In [ ]:
# Contrato compartido: df_raw, df_clean, X, y, particiones y escalador
# son producidos por las subsecciones de datos y preprocesamiento asignadas.
modelos_entrenados = {}
tipo_entrada = {}
NOMBRES_MODELOS = (
    "Regresión logística", "KNN", "SVM", "Árbol de decisión", "Random Forest"
)
print("Registros de modelos inicializados. Convención objetivo: 1=M (maligno), 0=B (benigno).")

## 1. Conjunto de datos

### 1.1. Carga del conjunto de datos

**Responsable: Poma.**

**Qué debes hacer**

- Comprueba si existe `/content/dataset/data.csv` y carga el CSV original del ejercicio en un DataFrame `df_raw`.
- Si el CSV no está disponible, utiliza el dataset Breast Cancer Wisconsin (Diagnostic) integrado en scikit-learn (`load_breast_cancer`) y genera un `df_raw` con predictores y una columna `diagnosis` que contenga etiquetas `M` y `B`.
- Muestra las primeras cinco filas, el número de registros y columnas y una nota visible que identifique la fuente utilizada (`CSV` o `sklearn`).

**Resultado esperado / criterio de aceptación**

Al terminar, `df_raw` existe, contiene una columna de diagnóstico y representa **una sola fuente** de datos.

**Atención:** No se debe obtener el diagnóstico de una variable predictora. Si se usa sklearn, recuerda que el target original tiene codificación inversa.

In [ ]:
# 1.1 Carga del CSV original (o alternativa reproducible de sklearn)
from pathlib import Path
import pandas as pd
from IPython.display import display

# En Colab se prefiere /content/dataset/data.csv; el segundo path permite
# ejecutar desde la carpeta Semana 13 del repositorio.
rutas_csv = [
    Path("/content/dataset/data.csv"),
    Path("dataset/data.csv"),
    Path("Semana 13/dataset/data.csv"),
]
ruta_csv = next((ruta for ruta in rutas_csv if ruta.is_file()), None)

if ruta_csv is not None:
    df_raw = pd.read_csv(ruta_csv)
    fuente_datos = f"CSV: {ruta_csv}"
else:
    from sklearn.datasets import load_breast_cancer
    datos_bc = load_breast_cancer(as_frame=True)
    df_raw = datos_bc.data.copy()
    # sklearn codifica 0=maligno, 1=benigno; convertir a etiquetas explícitas.
    df_raw["diagnosis"] = datos_bc.target.map({0: "M", 1: "B"})
    fuente_datos = "sklearn.datasets.load_breast_cancer (alternativa; 0 original=M, 1 original=B)"

if "diagnosis" not in df_raw.columns:
    raise ValueError("El conjunto debe contener una columna 'diagnosis'.")

print(f"Fuente utilizada: {fuente_datos}")
print(f"Dimensiones originales: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
display(df_raw.head())


### 1.2. Validación inicial y limpieza

**Responsable: Poma.**

**Qué debes hacer**

- Revisa tipos de datos, duplicados, valores nulos y distribución de las clases.
- Elimina columnas que no deben ser predictoras (`id`, `Unnamed: 32` u otras columnas enteramente vacías, si aparecen). Conserva `diagnosis` para formar el objetivo.
- Si existen ausentes, explica el criterio aplicado. Si se requiere imputación estadística, **no** calcules parámetros con el conjunto completo: esa imputación se debe ajustar usando únicamente entrenamiento en la sección 3.
- Guarda el resultado tabular en `df_clean`, sin modificar permanentemente `df_raw`.

**Resultado esperado / criterio de aceptación**

`df_clean` contiene `diagnosis` más las variables predictoras válidas. Reporta el antes/después y las decisiones de limpieza.

**Atención:** En caso de duplicados, justificar su tratamiento; no eliminar muestras automáticamente sin documentar por qué.

In [ ]:
# 1.2 Validación inicial y limpieza sin alterar df_raw
import pandas as pd
from IPython.display import display

print("Dimensiones antes de limpiar:", df_raw.shape)
print("Tipos de datos originales:")
display(df_raw.dtypes.rename("tipo").to_frame())
print("Filas duplicadas exactas:", int(df_raw.duplicated().sum()))
print("Valores ausentes por columna (solo columnas con ausentes):")
ausentes = df_raw.isna().sum()
display(ausentes[ausentes.gt(0)].rename("ausentes").to_frame())
print("Distribución original de diagnosis:")
display(df_raw["diagnosis"].value_counts(dropna=False).rename("conteo").to_frame())

# Se trabaja sobre una copia. Los identificadores y columnas de exportación vacías
# no describen características y no deben entrar a X.
df_clean = df_raw.copy()
df_clean.columns = [str(col).strip() for col in df_clean.columns]
columnas_no_predictoras = [
    col for col in df_clean.columns
    if col.lower() == "id"
    or col.lower().startswith("unnamed:")
    or (col.lower() != "diagnosis" and df_clean[col].isna().all())
]
df_clean = df_clean.drop(columns=columnas_no_predictoras)

if "diagnosis" not in df_clean.columns:
    raise ValueError("No se encontró la columna diagnosis después de la limpieza.")
df_clean["diagnosis"] = df_clean["diagnosis"].astype("string").str.strip().str.upper()
if df_clean["diagnosis"].isna().any():
    raise ValueError("Hay diagnósticos ausentes; no se eliminan filas de forma silenciosa.")

etiquetas_observadas = set(df_clean["diagnosis"].unique())
if not etiquetas_observadas.issubset({"M", "B"}):
    raise ValueError(f"Etiquetas de diagnóstico inesperadas: {sorted(etiquetas_observadas)}")

print("Columnas eliminadas (ID/exportación):", columnas_no_predictoras or "ninguna")
print("Filas duplicadas conservadas para no descartar muestras sin justificación:", int(df_clean.duplicated().sum()))
print("Dimensiones después de limpiar:", df_clean.shape)
print("Valores ausentes restantes:", int(df_clean.isna().sum().sum()))
print("Criterio de ausentes: no imputar antes de la partición; si hubiera ausentes en predictores, la imputación debe ajustarse solo con train en 3.2.")


### 1.3. Selección de X e y y codificación del diagnóstico

**Responsable: Poma.**

**Qué debes hacer**

- Crea `y` desde `df_clean["diagnosis"]` usando la convención `M → 1` y `B → 0`. Si el objetivo ya está numérico, valida explícitamente su significado antes de convertirlo.
- Crea `X` con **solo** las 30 características predictoras numéricas; excluye diagnóstico, identificadores y campos derivados de la etiqueta.
- Valida que `X` y `y` tienen el mismo número de filas, que `X` no incluye el diagnóstico, que `y` contiene únicamente `{0, 1}` y que no hay infinitos ni textos en `X`.
- Muestra forma de `X`, conteo de `y` y lista de columnas predictoras.

**Resultado esperado / criterio de aceptación**

Se pueden usar `X` e `y` en todas las siguientes secciones. Debe quedar claro que **1 = maligno**.

In [ ]:
# 1.3 Selección de predictores y codificación M=1 / B=0
import numpy as np
import pandas as pd

if "diagnosis" not in df_clean.columns:
    raise ValueError("df_clean debe contener diagnosis.")

# La etiqueta se mapea explícitamente; nunca se infiere desde una característica.
y = df_clean["diagnosis"].map({"M": 1, "B": 0}).astype("int64")
X = df_clean.drop(columns=["diagnosis"]).copy()

# El contrato del ejercicio requiere las 30 características numéricas originales.
columnas_no_numericas = X.select_dtypes(exclude=[np.number]).columns.tolist()
if columnas_no_numericas:
    raise TypeError(f"Predictores no numéricos detectados: {columnas_no_numericas}")
if X.shape[1] != 30:
    raise ValueError(f"Se esperaban 30 predictores; se encontraron {X.shape[1]}: {X.columns.tolist()}")

valores_X = X.to_numpy(dtype=float)
if len(X) != len(y):
    raise ValueError("X e y no tienen el mismo número de filas.")
if "diagnosis" in X.columns:
    raise AssertionError("La etiqueta no debe formar parte de X.")
if not set(y.unique()).issubset({0, 1}):
    raise ValueError("y debe contener únicamente 0 y 1.")
if np.isinf(valores_X).any():
    raise ValueError("X contiene valores infinitos; deben corregirse antes de entrenar.")

conteo_ausentes = X.isna().sum()
columnas_con_ausentes = conteo_ausentes[conteo_ausentes.gt(0)]
print("Codificación objetivo: 1 = M (maligno), 0 = B (benigno)")
print("Forma de X:", X.shape)
print(f"Valores ausentes en X: {int(conteo_ausentes.sum())}")
if not columnas_con_ausentes.empty:
    print("Ausentes por predictor:")
    display(columnas_con_ausentes.rename("ausentes").to_frame())
print("Los ausentes se reservan para imputación en 3.2, ajustada únicamente con X_train.")
print("Conteo de y (0=benigno, 1=maligno):")
display(y.value_counts().sort_index().rename(index={0: "0 = B", 1: "1 = M"}).rename("conteo").to_frame())
print("Predictores (30):")
print(X.columns.tolist())


## 2. Análisis exploratorio de datos (EDA)

### 2.1. Distribución de clases y balance del objetivo

**Responsable: Aldana.**

**Qué debes hacer**

- Usa `y` para calcular frecuencias absolutas y relativas de benignos (0) y malignos (1).
- Presenta un gráfico de barras con etiquetas legibles y porcentajes.
- Escribe una conclusión breve sobre la diferencia de proporciones y por qué puede influir en Accuracy y Recall.

**Resultado esperado / criterio de aceptación**

Tabla o impresión de conteos + figura + interpretación de 2–4 líneas. No modificar `X` ni `y`.

In [ ]:
# 2.1 — David Aldana Chavez: frecuencias absolutas y relativas del diagnóstico.
from IPython.display import display, Markdown

if not X.index.equals(y.index) or set(y.unique()) != {0, 1}:
    raise ValueError("EDA requiere X/y alineados y ambas clases: 0=B, 1=M.")
_etiquetas_eda = {0: "Benigno (B=0)", 1: "Maligno (M=1)"}
_colores_eda = {0: "#2878B5", 1: "#D95319"}
_conteos_eda = y.value_counts().reindex([0, 1], fill_value=0)
resumen_clases_eda = pd.DataFrame({
    "diagnostico": [_etiquetas_eda[c] for c in (0, 1)],
    "conteo": _conteos_eda.to_numpy(),
    "porcentaje": 100 * _conteos_eda.to_numpy() / len(y),
})
display(resumen_clases_eda.round(2))
fig, ax = plt.subplots(figsize=(7, 4))
_barras_eda = ax.bar(resumen_clases_eda["diagnostico"], resumen_clases_eda["conteo"],
                     color=[_colores_eda[c] for c in (0, 1)])
for barra, porcentaje in zip(_barras_eda, resumen_clases_eda["porcentaje"]):
    ax.annotate(f"{int(barra.get_height())} ({porcentaje:.1f}%)",
                (barra.get_x() + barra.get_width()/2, barra.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center")
ax.set(title="Distribución del diagnóstico — Breast Cancer", ylabel="Muestras", xlabel="Diagnóstico")
ax.set_ylim(0, _conteos_eda.max() * 1.18)
fig.tight_layout()
plt.show()
proporcion_maligna_eda = float(y.mean())
accuracy_mayoritaria_eda = float(_conteos_eda.max() / len(y))
display(Markdown(
    f"**Interpretación (Aldana):** se observan {int(_conteos_eda[0])} muestras benignas "
    f"y {int(_conteos_eda[1])} malignas ({100*proporcion_maligna_eda:.1f}% malignas). "
    f"Predecir siempre la clase mayoritaria alcanzaría Accuracy={accuracy_mayoritaria_eda:.3f}. "
    "Predecir siempre benigno tendría Recall_M=0: la exactitud por sí sola puede ocultar "
    "malignos no detectados. Se deben revisar Recall_M, F1_M y falsos negativos."))
# EDA descriptivo del conjunto disponible: no se modifica X/y, no se seleccionan
# variables ni hiperparámetros con estas salidas, que incluyen futuras filas de test.


### 2.2. Estadísticas descriptivas y distribución de predictores

**Responsable: Aldana.**

**Qué debes hacer**

- Muestra estadísticas descriptivas de `X` e inspecciona rangos y posibles valores extremos.
- Genera histogramas y/o boxplots de **un conjunto razonado** de variables (por ejemplo, radio, área, textura y concavidad), sin saturar el notebook con 30 gráficos redundantes.
- Compara visualmente al menos dos variables entre diagnósticos benigno y maligno, manteniendo la convención de colores/etiquetas en cada gráfico.
- Registra hallazgos observables, sin afirmar causalidad.

**Resultado esperado / criterio de aceptación**

Estadísticas descriptivas, gráficos legibles y conclusiones breves sobre variabilidad y diferencias visuales.

In [ ]:
# 2.2 — David Aldana Chavez: rangos, distribuciones y diferencias entre diagnósticos.
# Resolver ambas fuentes permitidas: nombres del CSV y nombres de sklearn.
def _columna_eda(nombre_csv, nombre_sklearn):
    for nombre in (nombre_csv, nombre_sklearn):
        if nombre in X.columns:
            return nombre
    raise KeyError(f"No se encontró {nombre_csv} ni {nombre_sklearn} en X.")

variables_eda = [
    _columna_eda("radius_mean", "mean radius"),
    _columna_eda("area_mean", "mean area"),
    _columna_eda("texture_mean", "mean texture"),
    _columna_eda("concavity_mean", "mean concavity"),
]
resumen_predictoras_eda = X.describe().T
_q1_eda, _q3_eda = X.quantile(0.25), X.quantile(0.75)
_iqr_eda = _q3_eda - _q1_eda
_extremos_eda = X.lt(_q1_eda - 1.5*_iqr_eda) | X.gt(_q3_eda + 1.5*_iqr_eda)
resumen_predictoras_eda["ausentes"] = X.isna().sum()
resumen_predictoras_eda["fuera_1_5_IQR"] = _extremos_eda.sum()
display(resumen_predictoras_eda.round(4))

# Radio y área describen tamaño; textura y concavidad aportan otras propiedades.
# Se conservan unidades originales, sin mezclar escalas en un mismo eje vertical.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, columna in zip(axes.ravel(), variables_eda):
    _valores_eda = X[columna].dropna()
    if _valores_eda.empty:
        ax.set_title(f"{columna}: sin valores observados")
        continue
    _bins_eda = np.histogram_bin_edges(_valores_eda, bins=20)
    for clase in (0, 1):
        valores = X.loc[y.eq(clase), columna].dropna()
        if not valores.empty:
            ax.hist(valores, bins=_bins_eda, density=True, alpha=0.5,
                    color=_colores_eda[clase], label=_etiquetas_eda[clase])
    ax.set(title=columna, xlabel="Valor original", ylabel="Densidad por clase")
    ax.legend(fontsize=8)
fig.suptitle("Distribuciones por diagnóstico — mismas clases y colores", y=1.01)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, columna in zip(axes, variables_eda[:2]):
    ax.boxplot([X.loc[y.eq(c), columna].dropna().to_numpy() for c in (0, 1)])
    ax.set_xticks([1, 2], [_etiquetas_eda[c] for c in (0, 1)])
    ax.set(title=columna, ylabel="Valor original")
fig.tight_layout()
plt.show()

medianas_por_diagnostico_eda = X[variables_eda].groupby(y).median().reindex([0, 1])
display(medianas_por_diagnostico_eda.rename(index=_etiquetas_eda).round(4))
_hallazgos_distribucion_eda = []
for columna in variables_eda:
    b, m = medianas_por_diagnostico_eda.loc[0, columna], medianas_por_diagnostico_eda.loc[1, columna]
    _hallazgos_distribucion_eda.append(f"{columna}: mediana B={b:.4g}, M={m:.4g}")
display(Markdown(
    "**Hallazgos observados (Aldana):** " + "; ".join(_hallazgos_distribucion_eda) + ". "
    "Los histogramas permiten inspeccionar separación y solapamiento entre diagnósticos; "
    "diferencias de medianas no implican separación perfecta ni causalidad. Los rangos "
    "dependen de la unidad de cada predictor, lo que motiva escalar antes de modelos "
    "sensibles a la magnitud. Los puntos fuera de 1.5 IQR son candidatos a valores extremos, "
    "no errores demostrados: se conservan sin recorte ni imputación en el EDA."))


### 2.3. Correlación y posibles redundancias

**Responsable: Aldana.**

**Qué debes hacer**

- Calcula la matriz de correlación de las 30 características de `X`.
- Muestra un mapa de calor legible (completo o filtrado a correlaciones altas) e identifica algunos pares de variables correlacionadas.
- Explica cómo la multicolinealidad puede afectar la interpretación de algunos clasificadores, y aclara que correlación **no** implica causalidad.
- No elimines columnas todavía sin acordarlo con Poma y Brickzon.

**Resultado esperado / criterio de aceptación**

Visualización de correlación y explicación fundamentada de 2–4 hallazgos.

**Atención:** El EDA es exploratorio. Cualquier selección automática de variables dependiente de datos deberá ajustarse en entrenamiento, nunca usando test.

In [ ]:
# 2.3 — David Aldana Chavez: correlación de las 30 características, sin eliminarlas.
correlacion_eda = X.corr(method="pearson")
fig, ax = plt.subplots(figsize=(14, 12))
_mascara_eda = np.triu(np.ones(correlacion_eda.shape, dtype=bool))
sns.heatmap(correlacion_eda, mask=_mascara_eda, cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={"label": "Correlación de Pearson"})
ax.set_title("Correlación entre predictores — triángulo inferior")
ax.tick_params(axis="both", labelsize=7)
plt.setp(ax.get_xticklabels(), rotation=90)
plt.setp(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
plt.show()

_triangulo_eda = np.triu(np.ones(correlacion_eda.shape, dtype=bool), k=1)
pares_correlacion_eda = (
    correlacion_eda.where(_triangulo_eda).stack().rename("correlacion")
    .rename_axis(["variable_1", "variable_2"]).reset_index()
)
pares_correlacion_eda["correlacion_absoluta"] = pares_correlacion_eda["correlacion"].abs()
pares_correlacion_eda = pares_correlacion_eda.sort_values(
    "correlacion_absoluta", ascending=False, kind="stable").reset_index(drop=True)
print("Diez pares con mayor correlación absoluta; sin diagonal ni pares duplicados:")
display(pares_correlacion_eda.head(10).round(4))
_pares_altos_eda = pares_correlacion_eda[pares_correlacion_eda["correlacion_absoluta"].ge(0.90)]
_ejemplos_corr_eda = "; ".join(
    f"{fila.variable_1} / {fila.variable_2}: r={fila.correlacion:.3f}"
    for fila in pares_correlacion_eda.head(3).itertuples()
)
display(Markdown(
    f"**Interpretación (Aldana):** hay {len(_pares_altos_eda)} pares con |r|≥0.90. "
    f"Los tres de mayor asociación son {_ejemplos_corr_eda}. "
    "Esto sugiere información compartida y posible redundancia. La multicolinealidad "
    "puede volver inestables los coeficientes individuales de regresión logística; "
    "su regularización L2 ayuda a estabilizar el ajuste, pero no convierte un "
    "coeficiente en un efecto causal. Pearson describe asociación lineal: una "
    "correlación pequeña no descarta relaciones no lineales. No se eliminan columnas; "
    "cualquier selección futura debe aprenderse solo dentro de entrenamiento."))


## 3. Modificaciones y preprocesamiento

### 3.1. División estratificada entrenamiento/prueba

**Responsable: Brickzon.**

**Qué debes hacer**

- Utiliza `train_test_split` con `test_size=0.20`, `random_state=RANDOM_STATE` y `stratify=y`.
- Crea **exactamente** `X_train`, `X_test`, `y_train`, `y_test`.
- Verifica tamaños y porcentajes de M/B en ambos subconjuntos. No alteres el test después de la división.

**Resultado esperado / criterio de aceptación**

Partición reproducible y estratificada; el conjunto de prueba se reserva para la evaluación final.

**Atención:** No escalar, imputar mediante estadísticas ni remuestrear antes de separar: eso causaría fuga de datos.

### 3.2. Escalamiento sin fuga de información

**Responsable: Brickzon.**

**Qué debes hacer**

- Elige `StandardScaler` como transformación inicial para regresión logística, KNN y SVM. Explica brevemente por qué estos modelos pueden ser sensibles a escalas distintas.
- Ajusta `escalador` con `fit` **solo** sobre `X_train`, y transforma ambos subconjuntos en `X_train_scaled` y `X_test_scaled` sin cambiar el orden ni la cantidad de filas.
- Conserva `X_train` y `X_test` sin escalar para Árbol de decisión y Random Forest.
- Si se necesita imputación, incorpora correctamente la imputación ajustada solo en train; idealmente construye un `Pipeline` coherente y evita cambiar el contrato de variables sin avisar.

**Resultado esperado / criterio de aceptación**

Existen `escalador`, `X_train_scaled` y `X_test_scaled`, compatibles con los modelos y sin uso del test para ajustar parámetros.

**Atención:** No es necesario implementar cuatro escaladores en este Ejercicio 3: es una propuesta independiente del Ejercicio 1.

### 3.3. Comprobaciones de preparación antes de entrenar

**Responsable: Brickzon.**

**Qué debes hacer**

- Comprueba que `X_train_scaled` y `X_test_scaled` mantienen los mismos números de filas y columnas que sus equivalentes originales.
- Verifica ausencia de NaN e infinitos en los arreglos que se usarán para entrenar y predecir.
- Comprueba que las etiquetas de ambas particiones respetan `1 = M`.
- Incluye una breve salida de verificación para indicar que ya se puede empezar la sección 4.

**Resultado esperado / criterio de aceptación**

Sin errores de forma, valores inválidos ni codificación de clases; todas las variables listas para los cinco clasificadores.

## 4. Modelos de clasificación

**Reglas comunes para los responsables de 4.1–4.5**

- Entrenar **solo** con la partición de entrenamiento. Usar `RANDOM_STATE` cuando el estimador lo permita.
- Mantener la clase **M = 1** como positiva.
- Registrar el modelo ya ajustado en `modelos_entrenados` con **el nombre exacto** establecido en 0.2, y el tipo de entrada en `tipo_entrada` (`"escalado"` u `"original"`).
- No reescribir `X_train`, `X_test`, `y_train`, `y_test` ni cambiar el escalador de otros integrantes.
- Mostrar una evidencia pequeña de entrenamiento correcto (por ejemplo, nombre del modelo, hiperparámetros principales o predicciones de ejemplo). **La evaluación comparativa del conjunto de prueba se centraliza en la sección 5**, no en cada modelo.
- Si se desean ajustar hiperparámetros, hacerlo solo con entrenamiento (validación cruzada/`GridSearchCV`), sin seleccionar modelos con el conjunto de prueba.

### 4.1. Regresión logística

**Responsable: Aldana.**

**Qué debes hacer**

- Instancia y entrena `LogisticRegression` usando `X_train_scaled` y `y_train`; configura un `max_iter` suficiente para evitar advertencias de convergencia.
- Registra el estimador bajo `modelos_entrenados["Regresión logística"]` y declara `tipo_entrada["Regresión logística"] = "escalado"`.
- Anota 1–2 líneas explicando por qué la regresión logística sirve como línea base interpretable.

**Resultado esperado / criterio de aceptación**

Regresión logística ajustada y registrada, lista para evaluar en la sección 5.

In [ ]:
# 4.1 — David Aldana Chavez: regresión logística con entrada escalada compartida.
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

_requeridas_lr = ("X_train", "y_train", "X_train_scaled", "modelos_entrenados", "tipo_entrada")
_faltantes_lr = [nombre for nombre in _requeridas_lr if nombre not in globals()]
if _faltantes_lr:
    raise RuntimeError("Completa 0.2 y 3.1–3.3 antes de 4.1. Faltan: " + ", ".join(_faltantes_lr))
if not X_train.index.equals(y_train.index) or set(y_train.unique()) != {0, 1}:
    raise ValueError("X_train/y_train deben estar alineados y respetar M=1/B=0.")
if np.shape(X_train_scaled) != X_train.shape or not np.isfinite(np.asarray(X_train_scaled)).all():
    raise ValueError("X_train_scaled debe conservar la forma de train y no contener NaN/inf.")

# C=1.0 es una configuración inicial fija; no se optimiza con el conjunto de prueba.
# lbfgs aplica regularización L2 por defecto. No se cambia el escalador de Brickzon.
regresion_logistica_aldana = LogisticRegression(
    C=1.0, solver="lbfgs", max_iter=5000, random_state=RANDOM_STATE,
)
# Una falta de convergencia detiene el registro; no se presenta como ajuste correcto.
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    regresion_logistica_aldana.fit(X_train_scaled, y_train)

modelos_entrenados["Regresión logística"] = regresion_logistica_aldana
tipo_entrada["Regresión logística"] = "escalado"
print("Regresión logística ajustada exclusivamente con entrenamiento escalado.")
print("C=1.0 | solver=lbfgs | max_iter=5000 | iteraciones:", regresion_logistica_aldana.n_iter_.tolist())
print("Clases del estimador:", regresion_logistica_aldana.classes_.tolist(), "(1=M)")
# Línea base: estima probabilidades y una frontera lineal con coeficientes
# interpretables de forma condicional. La correlación entre variables limita
# atribuir importancia individual. Predicciones y métricas de test se centralizan en 5.1.


### 4.2. K vecinos más cercanos (KNN)

**Responsable: Brickzon.**

**Qué debes hacer**

- Entrena un `KNeighborsClassifier` con los datos escalados; indica el número de vecinos `n_neighbors` elegido.
- Si comparas varios valores de K, selecciónalos con validación **solo sobre entrenamiento**.
- Registra `modelos_entrenados["KNN"]` y `tipo_entrada["KNN"] = "escalado"`.
- Explica en 1–2 líneas el efecto de la distancia y por qué debe escalarse la entrada.

**Resultado esperado / criterio de aceptación**

KNN entrenado y registrado, sin utilizar el conjunto de prueba para elegir K.

### 4.3. Máquinas de vectores de soporte (SVM)

**Responsable: Brickzon.**

**Qué debes hacer**

- Entrena un `SVC` con los datos escalados; documenta el kernel y al menos un hiperparámetro (por ejemplo, `C`).
- Para curva ROC, asegúrate de poder recuperar una puntuación de la **clase 1** mediante `decision_function` o, si corresponde, `predict_proba` (`probability=True`).
- Registra `modelos_entrenados["SVM"]` y `tipo_entrada["SVM"] = "escalado"`.

**Resultado esperado / criterio de aceptación**

SVM entrenado, puntuación para ROC identificada y registro consistente con el resto.

### 4.4. Árbol de decisión

**Responsable: Vera.**

**Qué debes hacer**

- Entrena `DecisionTreeClassifier` con `X_train` **sin escalar** y `y_train`.
- Controla al menos un parámetro de complejidad (por ejemplo, `max_depth` o `min_samples_leaf`) y justifica brevemente el criterio para limitar sobreajuste.
- Registra `modelos_entrenados["Árbol de decisión"]` y `tipo_entrada["Árbol de decisión"] = "original"`.

**Resultado esperado / criterio de aceptación**

Árbol ajustado, registrado y con control explícito de complejidad.

### 4.5. Random Forest

**Responsable: Vera.**

**Qué debes hacer**

- Entrena `RandomForestClassifier` con `X_train` sin escalar y `y_train`.
- Documenta `n_estimators` y, de usarlo, algún parámetro de regularización o profundidad.
- Registra `modelos_entrenados["Random Forest"]` y `tipo_entrada["Random Forest"] = "original"`.
- Confirma que el estimador expone `feature_importances_` para la interpretación de 6.1.

**Resultado esperado / criterio de aceptación**

Random Forest entrenado y registrado; quedan disponibles sus importancias de características.

## 5. Evaluación de los modelos

### 5.1. Predicciones y cálculo centralizado de métricas

**Responsable: Marco.**

**Qué debes hacer**

- Comprueba que `modelos_entrenados` contiene exactamente los cinco nombres acordados y que `tipo_entrada` tiene una entrada para cada uno.
- Para cada modelo selecciona `X_test_scaled` si su tipo es `"escalado"`, o `X_test` si es `"original"`; genera su predicción y una puntuación para ROC-AUC referida a `M = 1`.
- Calcula por modelo `Accuracy`, `Precision_M`, `Recall_M`, `F1_M` y `ROC_AUC` (usa `pos_label=1` cuando sea relevante y considera `zero_division=0` si corresponde).
- Guarda `resultados_df` con columnas `Modelo`, `Accuracy`, `Precision_M`, `Recall_M`, `F1_M`, `ROC_AUC` y define `y_pred_por_modelo`, `y_score_por_modelo` como diccionarios de resultados por nombre.
- Para el ROC-AUC usa **puntuaciones o probabilidades** de la clase maligna, nunca las etiquetas 0/1 de `predict` como sustituto de una puntuación.

**Resultado esperado / criterio de aceptación**

Existen `resultados_df`, `y_pred_por_modelo`, `y_score_por_modelo`; se ha evaluado **solo una vez** el test reservado y las cinco filas son comparables.

**Atención:** Para `predict_proba`, verificar el orden con `modelo.classes_`: no suponer que una columna fija corresponde siempre a maligno.

In [ ]:
NOMBRES_MODELOS = (
    "Regresión logística", "KNN", "SVM", "Árbol de decisión", "Random Forest"
)
COLUMNAS_RESULTADOS = ["Modelo", "Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"]


def _validar_contrato_evaluacion():
    requeridas = ("modelos_entrenados", "tipo_entrada", "X_test", "X_test_scaled", "y_test")
    faltantes = [nombre for nombre in requeridas if nombre not in globals()]
    if faltantes:
        raise RuntimeError("Faltan variables compartidas para evaluar: " + ", ".join(faltantes))
    if not isinstance(modelos_entrenados, dict) or not isinstance(tipo_entrada, dict):
        raise TypeError("modelos_entrenados y tipo_entrada deben ser diccionarios.")
    faltan_modelos = [n for n in NOMBRES_MODELOS if n not in modelos_entrenados]
    extras = [n for n in modelos_entrenados if n not in NOMBRES_MODELOS]
    if faltan_modelos or extras:
        raise ValueError(f"Registro de modelos incorrecto. Faltan: {faltan_modelos}; extras: {extras}.")
    faltan_tipos = [n for n in NOMBRES_MODELOS if n not in tipo_entrada]
    extras_tipo = [n for n in tipo_entrada if n not in NOMBRES_MODELOS]
    if faltan_tipos or extras_tipo:
        raise ValueError(f"Registro de tipos incorrecto. Faltan: {faltan_tipos}; extras: {extras_tipo}.")
    invalidos = {n: tipo_entrada[n] for n in NOMBRES_MODELOS
                 if tipo_entrada[n] not in ("escalado", "original")}
    if invalidos:
        raise ValueError(f"Cada tipo debe ser 'escalado' u 'original'; valores inválidos: {invalidos}.")
    forma_original = np.shape(X_test)
    forma_escalada = np.shape(X_test_scaled)
    if len(forma_original) != 2 or len(forma_escalada) != 2:
        raise ValueError("X_test y X_test_scaled deben ser arreglos bidimensionales.")
    if forma_original != forma_escalada:
        raise ValueError("X_test y X_test_scaled deben tener las mismas dimensiones.")
    if forma_original[0] != len(y_test):
        raise ValueError("X_test, X_test_scaled y y_test deben tener la misma cantidad de filas.")
    clases_test = set(np.asarray(y_test).ravel().tolist())
    if not clases_test.issubset({0, 1}) or clases_test != {0, 1}:
        raise ValueError("y_test debe contener ambas clases codificadas como 0=B y 1=M.")


def _score_clase_maligna(modelo, X_eval):
    clases = np.asarray(getattr(modelo, "classes_", []))
    posiciones = np.flatnonzero(clases == 1)
    if len(posiciones) != 1:
        raise ValueError("El modelo debe exponer classes_ con la clase positiva 1=M.")
    if hasattr(modelo, "predict_proba"):
        probabilidades = np.asarray(modelo.predict_proba(X_eval))
        if probabilidades.ndim != 2 or probabilidades.shape[1] != len(clases):
            raise ValueError("predict_proba devolvió dimensiones incompatibles con classes_.")
        score = probabilidades[:, posiciones[0]]
    elif hasattr(modelo, "decision_function"):
        score = np.asarray(modelo.decision_function(X_eval))
        if score.ndim == 2:
            score = score[:, posiciones[0]]
        elif score.ndim == 1 and len(clases) == 2 and posiciones[0] == 0:
            # En SVC binario, score positivo corresponde a classes_[1].
            score = -score
        elif score.ndim != 1:
            raise ValueError("decision_function devolvió una forma no compatible.")
    else:
        raise TypeError("El modelo necesita predict_proba() o decision_function() para ROC-AUC.")
    score = np.asarray(score, dtype=float).ravel()
    if len(score) != len(y_test) or not np.isfinite(score).all():
        raise ValueError("Las puntuaciones ROC deben ser finitas y corresponder a todo y_test.")
    return score


_validar_contrato_evaluacion()
y_pred_por_modelo = {}
y_score_por_modelo = {}
filas_metricas = []

for nombre in NOMBRES_MODELOS:
    modelo = modelos_entrenados[nombre]
    X_eval = X_test_scaled if tipo_entrada[nombre] == "escalado" else X_test
    y_pred = np.asarray(modelo.predict(X_eval)).ravel()
    if len(y_pred) != len(y_test):
        raise ValueError(f"{nombre}: predict() no devolvió una predicción por fila de y_test.")
    clases_modelo = set(np.asarray(getattr(modelo, "classes_", [])).tolist())
    if clases_modelo != {0, 1} or not set(y_pred.tolist()).issubset({0, 1}):
        raise ValueError(f"{nombre}: classes_ debe ser {{0, 1}} y predict() debe respetar 0=B, 1=M.")
    score_m = _score_clase_maligna(modelo, X_eval)
    y_pred_por_modelo[nombre] = y_pred
    y_score_por_modelo[nombre] = score_m
    filas_metricas.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_M": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall_M": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1_M": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, score_m),
    })

resultados_df = pd.DataFrame(filas_metricas, columns=COLUMNAS_RESULTADOS)
print("Métricas calculadas sobre el mismo conjunto de prueba para los cinco modelos.")
display(resultados_df.round(4))

### 5.2. Tabla comparativa y ordenamiento

**Responsable: Poma.**

**Qué debes hacer**

- Muestra `resultados_df` redondeado de manera uniforme y ordénalo según `Recall_M` y como criterio de desempate `F1_M` o `ROC_AUC`.
- Destaca qué modelo obtiene el mejor valor por métrica, sin llamar mejor a un modelo solamente por Accuracy.
- Anota de forma breve si hay diferencias entre el mejor modelo por Recall y el mejor por ROC-AUC.

**Resultado esperado / criterio de aceptación**

Tabla clara de cinco modelos, comparación defendible y lectura correcta de la clase positiva.

In [ ]:
# 5.2 Tabla comparativa ordenada por Recall_M y luego F1_M
import numpy as np
import pandas as pd
from IPython.display import display

columnas_metricas = ["Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"]
columnas_requeridas = ["Modelo", *columnas_metricas]
if "resultados_df" not in globals():
    raise NameError("Ejecuta primero la sección 5.1 para crear resultados_df.")
if not set(columnas_requeridas).issubset(resultados_df.columns):
    raise ValueError(f"resultados_df debe incluir estas columnas: {columnas_requeridas}")

modelos_esperados = {
    "Regresión logística", "KNN", "SVM", "Árbol de decisión", "Random Forest"
}
if resultados_df["Modelo"].isna().any() or not resultados_df["Modelo"].map(lambda nombre: isinstance(nombre, str)).all():
    raise ValueError("La columna Modelo debe contener nombres de modelo no nulos como texto.")
if resultados_df["Modelo"].duplicated().any():
    repetidos = resultados_df.loc[resultados_df["Modelo"].duplicated(keep=False), "Modelo"].tolist()
    raise ValueError(f"resultados_df contiene modelos duplicados: {repetidos}")
modelos_observados = set(resultados_df["Modelo"])
if len(resultados_df) != 5 or modelos_observados != modelos_esperados:
    faltantes = modelos_esperados - modelos_observados
    inesperados = modelos_observados - modelos_esperados
    raise ValueError(
        "resultados_df debe contener exactamente los cinco clasificadores acordados. "
        f"Faltantes: {sorted(faltantes)}; inesperados: {sorted(inesperados)}; "
        f"filas: {len(resultados_df)}."
    )

comparacion = resultados_df[columnas_requeridas].copy()
comparacion[columnas_metricas] = comparacion[columnas_metricas].apply(pd.to_numeric, errors="raise")
if not np.isfinite(comparacion[columnas_metricas].to_numpy(dtype=float)).all():
    raise ValueError("Las métricas deben ser numéricas y finitas; se encontraron NaN o infinitos.")
comparacion = comparacion.sort_values(
    ["Recall_M", "F1_M", "ROC_AUC"], ascending=False, kind="mergesort"
).reset_index(drop=True)

print("Comparación de modelos (1 = maligno). Orden: Recall_M, F1_M y ROC_AUC descendentes.")
tabla_mostrada = comparacion.copy()
ganadores_por_metrica = {}
for metrica in columnas_metricas:
    maximo = comparacion[metrica].max()
    es_maximo = np.isclose(comparacion[metrica].to_numpy(dtype=float), maximo)
    ganadores_por_metrica[metrica] = comparacion.loc[es_maximo, "Modelo"].tolist()
    tabla_mostrada[metrica] = [
        f"{valor:.4f}*" if np.isclose(valor, maximo) else f"{valor:.4f}"
        for valor in comparacion[metrica]
    ]
display(tabla_mostrada)
print("* Mejor valor observado para esa métrica; se marcan todos los empates (np.isclose).")

print("Modelos líderes por métrica:")
for metrica, ganadores in ganadores_por_metrica.items():
    maximo = comparacion[metrica].max()
    print(f"- {metrica}: {', '.join(ganadores)} ({maximo:.4f})")

ganadores_recall = ganadores_por_metrica["Recall_M"]
ganadores_auc = ganadores_por_metrica["ROC_AUC"]
if set(ganadores_recall) == set(ganadores_auc):
    print(
        "Recall_M y ROC_AUC tienen el mismo grupo de líderes: "
        f"{', '.join(ganadores_recall)}."
    )
else:
    coinciden = [modelo for modelo in ganadores_recall if modelo in set(ganadores_auc)]
    print(
        f"Líderes de Recall_M: {', '.join(ganadores_recall)}; "
        f"líderes de ROC_AUC: {', '.join(ganadores_auc)}."
    )
    if coinciden:
        print(f"Líderes compartidos entre ambas métricas: {', '.join(coinciden)}.")
    else:
        print("No hay modelos empatados como líderes en ambas métricas.")


### 5.3. Matrices de confusión y falsos negativos

**Responsable: Vera.**

**Qué debes hacer**

- Para cada modelo, usa `y_test` y `y_pred_por_modelo[nombre]` para elaborar una matriz de confusión con orden de etiquetas `[0, 1]`.
- Etiqueta los ejes como valores reales y predichos, con `0=B` y `1=M`, de forma consistente.
- Identifica el cuadrante de **falsos negativos de malignidad** (real M=1, predicho B=0) y compara su cantidad en los cinco modelos.
- Incluye una interpretación breve de por qué los falsos negativos son relevantes para este ejercicio.

**Resultado esperado / criterio de aceptación**

Cinco matrices de confusión legibles y comparación explícita de falsos negativos para la clase maligna.

### 5.4. Curvas ROC comparadas

**Responsable: Aldana.**

**Qué debes hacer**

- Utiliza las puntuaciones registradas en `y_score_por_modelo` y las etiquetas verdaderas `y_test` para construir una curva ROC por clasificador.
- Incluye la diagonal de azar, leyenda legible y el AUC de cada modelo. Si es viable, unifica las cinco curvas en un mismo gráfico para comparar.
- Explica en pocas líneas que ROC-AUC evalúa discriminación a varios umbrales y que no reemplaza el análisis de Recall_M ni falsos negativos.

**Resultado esperado / criterio de aceptación**

Figura ROC reproducible de los cinco modelos y explicación coherente de sus AUC.

**Atención:** No usar `y_pred` binario como entrada a `roc_curve` cuando existen scores/probabilidades.

In [ ]:
# 5.4 — David Aldana Chavez: cinco curvas ROC desde los scores comunes de 5.1.
_requeridas_roc = ("y_score_por_modelo", "y_test", "NOMBRES_MODELOS")
_faltantes_roc = [nombre for nombre in _requeridas_roc if nombre not in globals()]
if _faltantes_roc:
    raise RuntimeError("Ejecuta 5.1 antes de 5.4. Faltan: " + ", ".join(_faltantes_roc))
_y_roc = np.asarray(y_test)
if _y_roc.ndim != 1 or set(np.unique(_y_roc)) != {0, 1}:
    raise ValueError("y_test debe ser un vector con ambas clases, 1=M y 0=B.")
_faltan_scores_roc = set(NOMBRES_MODELOS) - set(y_score_por_modelo)
if _faltan_scores_roc:
    raise ValueError(f"Faltan puntuaciones para las cinco curvas: {sorted(_faltan_scores_roc)}")

curvas_roc_aldana = {}
_filas_auc_aldana = []
fig, ax = plt.subplots(figsize=(8, 6))
for nombre, color in zip(NOMBRES_MODELOS, sns.color_palette("colorblind", len(NOMBRES_MODELOS))):
    # Son probabilidades de M o decision_function orientada a M por 5.1.
    # No se usa y_pred binario ni se vuelve a ajustar ningún estimador.
    score = y_score_por_modelo[nombre]
    if isinstance(score, pd.Series) and not score.index.equals(y_test.index):
        raise ValueError(f"{nombre}: índices de los scores desalineados con y_test.")
    score = np.asarray(score, dtype=float)
    if score.ndim != 1 or len(score) != len(_y_roc) or not np.isfinite(score).all():
        raise ValueError(f"{nombre}: se requiere un score finito por muestra de test.")
    fpr, tpr, umbrales = roc_curve(_y_roc, score, pos_label=1)
    auc_observado = float(roc_auc_score(_y_roc, score))
    curvas_roc_aldana[nombre] = {"fpr": fpr, "tpr": tpr, "umbrales": umbrales, "auc": auc_observado}
    _filas_auc_aldana.append({"Modelo": nombre, "ROC_AUC": auc_observado})
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{nombre} (AUC={auc_observado:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Azar (AUC=0.500)")
ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="Tasa de falsos positivos (1 − especificidad)",
       ylabel="Tasa de verdaderos positivos (Recall_M)", title="Curvas ROC — clase positiva: maligno (M=1)")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()
auc_roc_aldana = pd.DataFrame(_filas_auc_aldana)
display(auc_roc_aldana.round(4))
display(Markdown(
    "**Interpretación (Aldana):** la tabla y la leyenda muestran el AUC observado de "
    "los cinco modelos sobre el mismo test. ROC-AUC resume discriminación a través "
    "de varios umbrales; 0.5 representa discriminación al azar y 1 separación perfecta "
    "en esta muestra. No mide calibración ni reemplaza Recall_M y falsos negativos "
    "del umbral utilizado en 5.1/5.3. No se selecciona un umbral ni se reajustan "
    "hiperparámetros con estas curvas; diferencias pequeñas requieren validación adicional."))


### 5.5. Análisis de errores y elección del modelo

**Responsable: Marco.**

**Qué debes hacer**

- Cruza la tabla de métricas con las matrices de confusión: identifica cuál(es) detectan más casos M y cuál(es) generan menos falsos negativos.
- Propón un modelo preferido **para el ejercicio académico**, justificando el equilibrio entre Recall_M, Precision_M, F1_M y ROC_AUC y aclarando las limitaciones del conjunto de prueba pequeño.
- Explica por qué seleccionar según Accuracy exclusivamente puede ser inapropiado si la prioridad son las muestras malignas.
- No afirmar que el mejor modelo sirve para uso clínico real ni ajustar umbrales sobre el test para reportar el mismo test como validación imparcial.

**Resultado esperado / criterio de aceptación**

Selección argumentada y basada en resultados reales, sin sobreinterpretar el rendimiento.

In [ ]:
# Comparación académica basada en los resultados observados. Vera conserva la autoría
# y presentación de las matrices de confusión de 5.3.
requeridas_55 = ("resultados_df", "y_pred_por_modelo", "y_test")
faltantes_55 = [nombre for nombre in requeridas_55 if nombre not in globals()]
if faltantes_55:
    print("PENDIENTE: ejecutar 5.1 y disponer de y_test. Faltan: " + ", ".join(faltantes_55))
else:
    columnas_55 = {"Modelo", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"}
    if not columnas_55.issubset(resultados_df.columns):
        raise ValueError("resultados_df no contiene todas las métricas requeridas para 5.5.")
    if set(resultados_df["Modelo"].tolist()) != set(NOMBRES_MODELOS):
        raise ValueError("resultados_df debe incluir exactamente los cinco clasificadores acordados.")
    faltan_preds_55 = [n for n in NOMBRES_MODELOS if n not in y_pred_por_modelo]
    if faltan_preds_55:
        raise ValueError("Faltan predicciones de: " + ", ".join(faltan_preds_55))
    y_real = np.asarray(y_test).ravel()
    if not set(np.unique(y_real).tolist()).issubset({0, 1}) or set(np.unique(y_real)) != {0, 1}:
        raise ValueError("y_test debe codificar ambas clases: 0=B y 1=M.")
    conteo_fn = {}
    for nombre in NOMBRES_MODELOS:
        pred = np.asarray(y_pred_por_modelo[nombre]).ravel()
        if len(pred) != len(y_real):
            raise ValueError(f"{nombre}: predicciones y y_test tienen dimensiones distintas.")
        if not set(np.unique(pred).tolist()).issubset({0, 1}):
            raise ValueError(f"{nombre}: predicciones fuera de la codificación 0=B, 1=M.")
        conteo_fn[nombre] = int(np.sum((y_real == 1) & (pred == 0)))
    analisis_df = resultados_df.copy()
    analisis_df["Falsos_negativos_M"] = analisis_df["Modelo"].map(conteo_fn)
    if analisis_df["Falsos_negativos_M"].isna().any():
        raise ValueError("No fue posible asociar todos los conteos de falsos negativos a los modelos.")
    display(analisis_df.sort_values(
        ["Recall_M", "F1_M", "ROC_AUC"], ascending=[False, False, False], kind="stable"
    ).reset_index(drop=True).round(4))
    max_recall = analisis_df["Recall_M"].max()
    max_precision = analisis_df["Precision_M"].max()
    max_f1 = analisis_df["F1_M"].max()
    max_auc = analisis_df["ROC_AUC"].max()
    min_fn = int(analisis_df["Falsos_negativos_M"].min())
    print("Mayor Recall_M (más detección de malignidad): " + ", ".join(
        analisis_df.loc[analisis_df["Recall_M"] == max_recall, "Modelo"].tolist()))
    print("Menor cantidad de falsos negativos M: " + ", ".join(
        analisis_df.loc[analisis_df["Falsos_negativos_M"] == min_fn, "Modelo"].tolist()) + f" ({min_fn})")
    print("Mayor Precision_M: " + ", ".join(analisis_df.loc[analisis_df["Precision_M"] == max_precision, "Modelo"].tolist()))
    print("Mayor F1_M: " + ", ".join(analisis_df.loc[analisis_df["F1_M"] == max_f1, "Modelo"].tolist()))
    print("Mayor ROC_AUC: " + ", ".join(analisis_df.loc[analisis_df["ROC_AUC"] == max_auc, "Modelo"].tolist()))
    # Regla declarada: minimizar FN; desempates por Recall_M, F1_M y ROC_AUC.
    analisis_df = analisis_df.sort_values(
        ["Falsos_negativos_M", "Recall_M", "F1_M", "ROC_AUC"],
        ascending=[True, False, False, False], kind="stable"
    ).reset_index(drop=True)
    recomendado = analisis_df.iloc[0]
    print(f"Criterio académico: mínimo FN, luego máximo Recall_M, F1_M y ROC_AUC. "
          f"Recomendación para este test: {recomendado['Modelo']}.")
    print("Accuracy no debe ser el único criterio cuando interesa detectar malignidad: puede ocultar falsos negativos. "
          "Precision_M, Recall_M, F1_M, ROC_AUC y FN representan aspectos distintos. "
          "La recomendación describe esta partición académica y no implica utilidad clínica ni superioridad general.")

## 6. Interpretación y discusión

### 6.1. Importancia de características en Random Forest

**Responsable: Vera.**

**Qué debes hacer**

- Recupera `feature_importances_` del modelo `Random Forest` entrenado en 4.5 y vincúlalo con los nombres originales de `X.columns`.
- Muestra una gráfica horizontal con las 10 características de mayor importancia.
- Explica que las importancias internas del bosque indican contribución al modelo, **no causalidad** ni relevancia médica validada.

**Resultado esperado / criterio de aceptación**

Gráfico legible de top 10 de características y explicación breve de sus limitaciones.

### 6.2. Síntesis del EDA frente a los resultados

**Responsable: Aldana.**

**Qué debes hacer**

- Relaciona dos o tres hallazgos del EDA (balance, distribuciones o correlaciones) con lo observado en el rendimiento de los modelos.
- Distingue hechos observados de interpretaciones; evita afirmar que cierta variable causa malignidad.
- Enumera limitaciones metodológicas: tamaño del dataset, variabilidad de una partición train/test y ausencia de validación externa.

**Resultado esperado / criterio de aceptación**

Párrafo o lista breve de hallazgos y limitaciones respaldados por las salidas del notebook.

In [ ]:
# 6.2 — David Aldana Chavez: síntesis basada en las salidas del EDA y la evaluación.
_requeridas_sintesis = ("resumen_clases_eda", "medianas_por_diagnostico_eda", "pares_correlacion_eda")
_faltantes_sintesis = [nombre for nombre in _requeridas_sintesis if nombre not in globals()]
if _faltantes_sintesis:
    raise RuntimeError("Completa primero 2.1–2.3. Faltan: " + ", ".join(_faltantes_sintesis))

_par_principal_eda = pares_correlacion_eda.iloc[0]
_medianas_tamano_eda = "; ".join(
    f"{columna}: B={medianas_por_diagnostico_eda.loc[0, columna]:.4g}, "
    f"M={medianas_por_diagnostico_eda.loc[1, columna]:.4g}"
    for columna in variables_eda[:2]
)
_texto_sintesis_eda = [
    f"**Balance observado:** {100*proporcion_maligna_eda:.1f}% de muestras malignas. "
    f"La Accuracy de una predicción mayoritaria constante sería {accuracy_mayoritaria_eda:.3f}; "
    "este referente explica por qué la comparación también debe examinar Recall_M y FN.",
    f"**Distribuciones observadas:** medianas de tamaño, {_medianas_tamano_eda}. "
    "Estas diferencias describen asociaciones con el diagnóstico y motivan estudiar "
    "capacidad de discriminación; no prueban causalidad ni separación completa. "
    "El escalamiento permite comparar magnitudes en regresión logística, KNN y SVM.",
    f"**Correlación observada:** {_par_principal_eda['variable_1']} y "
    f"{_par_principal_eda['variable_2']} presentan r={_par_principal_eda['correlacion']:.3f}. "
    "La información compartida limita interpretar coeficientes como contribuciones "
    "independientes. La regularización de la regresión logística estabiliza el ajuste; "
    "el EDA no ha eliminado predictores ni optimizado hiperparámetros.",
]
if "resultados_df" not in globals() or "y_pred_por_modelo" not in globals():
    _texto_sintesis_eda.append(
        "**Rendimiento pendiente:** ejecutar 5.1 con los cinco clasificadores antes de "
        "vincular cifras de rendimiento con los hallazgos; no se presuponen resultados.")
else:
    _metricas_lr_eda = resultados_df[resultados_df["Modelo"].eq("Regresión logística")]
    if len(_metricas_lr_eda) != 1 or "Regresión logística" not in y_pred_por_modelo:
        raise ValueError("Faltan las métricas o predicciones únicas de regresión logística de 5.1.")
    _fila_lr_eda = _metricas_lr_eda.iloc[0]
    _real_sintesis = np.asarray(y_test)
    _pred_lr_eda = np.asarray(y_pred_por_modelo["Regresión logística"])
    if _pred_lr_eda.shape != _real_sintesis.shape:
        raise ValueError("Predicciones de regresión logística desalineadas con y_test.")
    _fn_lr_eda = int(((_real_sintesis == 1) & (_pred_lr_eda == 0)).sum())
    _texto_sintesis_eda.append(
        f"**Resultado observado:** regresión logística obtiene Accuracy={_fila_lr_eda['Accuracy']:.3f}, "
        f"Recall_M={_fila_lr_eda['Recall_M']:.3f}, F1_M={_fila_lr_eda['F1_M']:.3f}, "
        f"ROC-AUC={_fila_lr_eda['ROC_AUC']:.3f} y {_fn_lr_eda} falsos negativos. "
        "ROC-AUC cuantifica si las diferencias exploradas se traducen en discriminación "
        "fuera de entrenamiento; Recall_M y FN muestran los malignos no detectados al "
        "umbral usado. El rendimiento pertenece al conjunto de variables y al modelo: "
        "no permite atribuir el acierto a una característica aislada.")
_texto_sintesis_eda.append(
    f"**Limitaciones:** {len(X)} muestras, una sola partición entrenamiento/prueba, "
    "EDA descriptivo que incluye las futuras filas de test y ausencia de validación externa. "
    "Este EDA no se usa para seleccionar variables, modelos o umbrales. La interpretación "
    "comparativa del test es académica; no demuestra desempeño clínico ni generalización. "
    "Validación cruzada dentro de entrenamiento y validación externa son trabajo futuro.")
conclusion_eda_aldana = "\n\n".join(_texto_sintesis_eda)
display(Markdown(conclusion_eda_aldana))


### 6.3. Conclusiones, reproducibilidad y recomendaciones

**Responsable: Marco.**

**Qué debes hacer**

- Resume el objetivo, el método de separación, el escalamiento aplicado solo al entrenamiento y los cinco modelos comparados.
- Indica el mejor resultado **con valores numéricos que aparezcan realmente en `resultados_df`**, así como al menos una limitación.
- Propón como trabajo futuro validación cruzada, ajuste de hiperparámetros y evaluación externa; deja claro que son recomendaciones y no trabajos ya realizados.
- Verifica que todas las gráficas tienen títulos, etiquetas y que ninguna conclusión contradiga los valores observados.

**Resultado esperado / criterio de aceptación**

Conclusiones completas, no genéricas, coherentes con las evidencias del notebook.

In [ ]:
from IPython.display import display, Markdown

nombres_esperados = list(NOMBRES_MODELOS)
metricas_esperadas = ["Modelo", "Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"]
conclusiones_generadas = False
conclusion_modelo_recomendado = None
conclusion_metricas_resumen = None

if "resultados_df" not in globals() or "analisis_df" not in globals():
    display(Markdown("""### Conclusiones (pendientes de integración)

El objetivo es comparar regresión logística, KNN, SVM, árbol de decisión y Random Forest en la clasificación binaria del conjunto Breast Cancer Wisconsin (Diagnostic), con 1 = maligno y 0 = benigno. La metodología acordada contempla una partición estratificada de entrenamiento y prueba; el escalador se ajusta únicamente con entrenamiento y los modelos se comparan sobre el mismo conjunto de prueba.

**PENDIENTE:** aún no están disponibles las métricas y el análisis comparativo completos. No se atribuye rendimiento a ningún clasificador ni se muestran valores hasta integrar los cinco modelos reales.

La interpretación final estará limitada por el tamaño del conjunto de prueba y por una única partición. Como trabajo futuro se recomienda validación cruzada, optimización de hiperparámetros exclusivamente dentro de entrenamiento y validación externa con datos independientes; estas actividades son recomendaciones y no procedimientos realizados aquí.

Este experimento es académico y no constituye validación clínica."""))
else:
    if not set(metricas_esperadas).issubset(resultados_df.columns):
        raise ValueError("resultados_df no tiene el esquema requerido para redactar conclusiones.")
    if set(resultados_df["Modelo"].tolist()) != set(nombres_esperados):
        raise ValueError("Las conclusiones requieren resultados de los cinco modelos, exactamente una fila por modelo.")
    if "Falsos_negativos_M" not in analisis_df.columns or set(analisis_df["Modelo"]) != set(nombres_esperados):
        raise ValueError("Ejecute 5.5 y verifique los falsos negativos de los cinco modelos antes de concluir.")
    if analisis_df["Modelo"].duplicated().any() or resultados_df["Modelo"].duplicated().any():
        raise ValueError("Se esperaba una sola fila por clasificador en resultados y análisis.")
    fila = analisis_df.sort_values(
        ["Falsos_negativos_M", "Recall_M", "F1_M", "ROC_AUC"],
        ascending=[True, False, False, False], kind="stable"
    ).iloc[0]
    conclusion_modelo_recomendado = fila["Modelo"]
    conclusion_metricas_resumen = fila[["Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC", "Falsos_negativos_M"]].to_dict()
    conclusiones_generadas = True
    resultados = resultados_df.set_index("Modelo").loc[nombres_esperados]
    lineas = [
        "### Conclusiones, reproducibilidad y recomendaciones", "",
        "El objetivo fue comparar cinco clasificadores supervisados para distinguir muestras benignas (0) y malignas (1). La metodología utilizó una partición estratificada entrenamiento/prueba y reservó el ajuste del escalador para entrenamiento. Regresión logística, KNN y SVM reciben datos escalados; árbol de decisión y Random Forest reciben predictores originales. Los cinco se evalúan sobre el mismo conjunto de prueba.", "",
        f"Con el criterio académico declarado (mínimo de falsos negativos y desempate por Recall_M, F1_M y ROC_AUC), la recomendación para esta partición es **{fila['Modelo']}**: Precision_M={fila['Precision_M']:.3f}, Recall_M={fila['Recall_M']:.3f}, F1_M={fila['F1_M']:.3f}, ROC_AUC={fila['ROC_AUC']:.3f} y falsos negativos={int(fila['Falsos_negativos_M'])}.", "",
        "Métricas observadas (sin redondeo previo al cálculo):", "",
        "| Modelo | Accuracy | Precision_M | Recall_M | F1_M | ROC_AUC |", "|---|---:|---:|---:|---:|---:|"
    ]
    for nombre, metrica in resultados.iterrows():
        lineas.append(f"| {nombre} | {metrica['Accuracy']:.3f} | {metrica['Precision_M']:.3f} | {metrica['Recall_M']:.3f} | {metrica['F1_M']:.3f} | {metrica['ROC_AUC']:.3f} |")
    lineas += ["", "Las cifras describen esta partición y pueden variar con otra muestra o partición; un test acotado no demuestra generalización ni utilidad clínica.", "", "**Trabajo futuro recomendado, no ejecutado en este experimento:** validación cruzada, ajuste de hiperparámetros dentro de entrenamiento y validación externa sobre datos independientes."]
    display(Markdown("\n".join(lineas)))

## 7. Control final de integración

### 7.1. Verificación y entrega en Colab

**Responsable: Marco.**

**Qué debes hacer**

- Abre la versión final en Google Colab, usa **Reiniciar sesión y ejecutar todo** y comprueba que todas las celdas terminan sin excepción.
- Comprueba que en un entorno limpio se puede obtener `df_raw`, `X`, `y`, la división train/test, los cinco modelos, `resultados_df` y todas las figuras.
- Confirma que los resultados se han calculado con etiquetas 1=M y 0=B y que no hubo `fit` del escalador sobre test ni preprocesamiento con información del test.
- Revisa los nombres de responsables en celdas de texto, la ortografía, títulos, leyendas y que no haya salidas de errores ni celdas experimentales sobrantes.
- Registra de forma breve la fuente del dataset, las versiones principales de librerías si resulta útil, y el nombre del archivo final compartido con el grupo.

**Resultado esperado / criterio de aceptación**

Notebook listo para entregar: se ejecuta de arriba abajo en Colab y conserva evidencias de todas las etapas.

In [ ]:
# Controles observables de integración. Ausencia de dependencias => PENDIENTE;
# una condición evaluada e incorrecta => FAIL.
def _reportar_estado(etiqueta, estado, detalle=""):
    if estado not in ("PASS", "FAIL", "PENDIENTE"):
        raise ValueError("Estado de verificación desconocido.")
    print(f"{estado}: {etiqueta}" + (f" — {detalle}" if detalle else ""))

compartidas = ["df_raw", "df_clean", "X", "y", "X_train", "X_test", "y_train", "y_test",
               "escalador", "X_train_scaled", "X_test_scaled"]
for nombre in compartidas:
    _reportar_estado(f"variable compartida {nombre}", "PASS" if nombre in globals() else "PENDIENTE",
                     "presente" if nombre in globals() else "aún no integrada")

if "sys" in globals():
    _reportar_estado("intérprete Python 3", "PASS" if sys.version_info.major == 3 else "FAIL",
                     sys.version.split()[0])
else:
    _reportar_estado("intérprete Python 3", "PENDIENTE", "no se pudo consultar sys.version_info")

if "RANDOM_STATE" in globals():
    _reportar_estado("RANDOM_STATE configurado en 42", "PASS" if RANDOM_STATE == 42 else "FAIL", str(RANDOM_STATE))
else:
    _reportar_estado("RANDOM_STATE configurado en 42", "PENDIENTE", "falta ejecutar 0.1")

if "modelos_entrenados" not in globals() or "tipo_entrada" not in globals():
    _reportar_estado("registro y tipos de los cinco modelos", "PENDIENTE", "faltan contenedores compartidos")
else:
    faltantes_modelos = [n for n in NOMBRES_MODELOS if n not in modelos_entrenados]
    faltantes_tipos = [n for n in NOMBRES_MODELOS if n not in tipo_entrada]
    extras_modelos = [n for n in modelos_entrenados if n not in NOMBRES_MODELOS]
    tipos_invalidos = {n: tipo_entrada[n] for n in tipo_entrada if n in NOMBRES_MODELOS and tipo_entrada[n] not in ("escalado", "original")}
    if extras_modelos or tipos_invalidos:
        _reportar_estado("registro y tipos de los cinco modelos", "FAIL", f"extras={extras_modelos}; tipos inválidos={tipos_invalidos}")
    elif faltantes_modelos or faltantes_tipos:
        _reportar_estado("registro y tipos de los cinco modelos", "PENDIENTE", f"faltan modelos={faltantes_modelos}; faltan tipos={faltantes_tipos}")
    else:
        _reportar_estado("registro y tipos de los cinco modelos", "PASS", "nombres exactos y valores de tipo válidos")

partes = ("X_train", "X_test", "y_train", "y_test", "X_train_scaled", "X_test_scaled")
if not all(nombre in globals() for nombre in partes):
    _reportar_estado("dimensiones train/test originales y escaladas", "PENDIENTE", "faltan particiones o matrices escaladas")
else:
    try:
        formas = {nombre: np.shape(globals()[nombre]) for nombre in partes}
        vectores = ("y_train", "y_test")
        matrices = ("X_train", "X_test", "X_train_scaled", "X_test_scaled")
        if any(len(formas[nombre]) != 1 for nombre in vectores):
            raise ValueError("y_train y y_test deben ser vectores unidimensionales")
        if any(len(formas[nombre]) != 2 for nombre in matrices):
            raise ValueError("las matrices X deben ser bidimensionales")
        dimensiones_ok = (
            formas["X_train"][0] == formas["y_train"][0]
            and formas["X_test"][0] == formas["y_test"][0]
            and formas["X_train"] == formas["X_train_scaled"]
            and formas["X_test"] == formas["X_test_scaled"]
            and formas["X_train"][1] == formas["X_test"][1]
        )
        _reportar_estado("dimensiones train/test originales y escaladas", "PASS" if dimensiones_ok else "FAIL",
                         f"formas={formas}")
    except Exception as error_forma:
        _reportar_estado("dimensiones train/test originales y escaladas", "FAIL",
                         f"dimensión inválida: {type(error_forma).__name__}: {error_forma}")

if "y" not in globals():
    _reportar_estado("codificación 1=M y 0=B", "PENDIENTE", "falta y")
else:
    valores_y = set(np.asarray(y).ravel().tolist())
    if not valores_y.issubset({0, 1}):
        _reportar_estado("codificación 1=M y 0=B", "FAIL", f"valores observados={valores_y}")
    elif "df_clean" not in globals() or "diagnosis" not in getattr(df_clean, "columns", []):
        _reportar_estado("significado semántico de las etiquetas", "PENDIENTE", "no hay diagnosis de referencia para comprobar M→1/B→0")
    else:
        diagnosis = df_clean["diagnosis"].astype(str).str.upper().to_numpy()
        etiquetas = np.asarray(y).ravel()
        semantica_ok = (len(diagnosis) == len(etiquetas) and
                        np.array_equal(etiquetas, np.where(diagnosis == "M", 1, np.where(diagnosis == "B", 0, -1))))
        _reportar_estado("codificación 1=M y 0=B contrastada con diagnosis", "PASS" if semantica_ok else "FAIL")

# Indicador parcial: la cantidad de filas vista por el escalador debe coincidir
# con train, pero esto no demuestra por sí sola sobre qué datos se llamó fit().
if "escalador" not in globals() or "X_train" not in globals():
    _reportar_estado("indicador parcial: n_samples_seen_ frente a filas de train", "PENDIENTE", "faltan escalador o X_train")
else:
    try:
        observadas = getattr(escalador, "n_samples_seen_", None)
        if observadas is None:
            _reportar_estado("indicador parcial: n_samples_seen_ frente a filas de train", "PENDIENTE", "el escalador no expone n_samples_seen_")
        else:
            coincide_train = bool(np.all(np.asarray(observadas) == len(X_train)))
            _reportar_estado("indicador parcial: n_samples_seen_ coincide con filas de train",
                             "PASS" if coincide_train else "FAIL",
                             "esta coincidencia no prueba que fit() usara exclusivamente train")
    except Exception as error_escalador:
        _reportar_estado("indicador parcial: n_samples_seen_ frente a filas de train", "FAIL",
                         f"no se pudo comprobar: {type(error_escalador).__name__}: {error_escalador}")

# Búsqueda limitada de llamadas directas; es evidencia parcial, no auditoría completa.
import re
historial_codigo = globals().get("In") or globals().get("_ih")
if historial_codigo is not None and len(historial_codigo) > 1:
    codigo_ejecutado = "\n".join(historial_codigo[1:-1])
    fit_directo_en_test = bool(re.search(r"\.fit(?:_transform)?\s*\(\s*X_test(?:_scaled)?\b", codigo_ejecutado))
    _reportar_estado("búsqueda parcial de llamadas directas fit(X_test)",
                     "FAIL" if fit_directo_en_test else "PASS",
                     "un resultado negativo no descarta otras formas de fuga")
else:
    _reportar_estado("búsqueda parcial de llamadas directas fit(X_test)", "PENDIENTE", "historial de código no disponible")

# La auditoría concluyente requiere revisar el flujo completo de carga, partición,
# transformaciones, selección y entrenamiento; estos indicadores no la sustituyen.
_reportar_estado("auditoría concluyente de ausencia de fuga de datos", "PENDIENTE",
                 "requiere revisar el flujo integrado y su ejecución desde sesión limpia")

if not all(nombre in globals() for nombre in ("resultados_df", "y_pred_por_modelo", "y_score_por_modelo", "y_test")):
    _reportar_estado("métricas, predicciones y scores ROC", "PENDIENTE", "falta ejecutar 5.1 con los cinco modelos")
else:
    esquema_ok = list(resultados_df.columns) == metricas_esperadas if "metricas_esperadas" in globals() else list(resultados_df.columns) == ["Modelo", "Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"]
    contenido_ok = (set(resultados_df["Modelo"]) == set(NOMBRES_MODELOS)
                    and all(n in y_pred_por_modelo and n in y_score_por_modelo for n in NOMBRES_MODELOS)
                    and all(len(y_pred_por_modelo[n]) == len(y_test) and len(y_score_por_modelo[n]) == len(y_test) for n in NOMBRES_MODELOS))
    if not esquema_ok or not contenido_ok:
        _reportar_estado("resultados y predicciones del conjunto de prueba", "FAIL", "esquema, modelos o longitudes inconsistentes")
    else:
        _reportar_estado("resultados y predicciones del conjunto de prueba", "PASS", "cinco modelos con longitudes compatibles")

if ("analisis_df" not in globals() or "Falsos_negativos_M" not in getattr(analisis_df, "columns", [])
        or "y_test" not in globals() or "y_pred_por_modelo" not in globals()):
    _reportar_estado("selección y conclusiones coherentes con valores reales", "PENDIENTE", "requiere resultados, predicciones y ejecución de 5.5/6.3")
else:
    conteos_recalculados = {
        n: int(np.sum((np.asarray(y_test).ravel() == 1) & (np.asarray(y_pred_por_modelo[n]).ravel() == 0)))
        for n in NOMBRES_MODELOS if n in y_pred_por_modelo
    }
    conteos_registrados = analisis_df.set_index("Modelo")["Falsos_negativos_M"].to_dict()
    analisis_coherente = (set(conteos_recalculados) == set(NOMBRES_MODELOS)
                          and all(conteos_registrados.get(n) == conteos_recalculados[n] for n in conteos_recalculados))
    if not analisis_coherente:
        _reportar_estado("falsos negativos en 5.5 coinciden con y_test/predicciones", "FAIL")
    else:
        _reportar_estado("falsos negativos en 5.5 coinciden con y_test/predicciones", "PASS")
    seleccion_esperada = analisis_df.sort_values(
        ["Falsos_negativos_M", "Recall_M", "F1_M", "ROC_AUC"],
        ascending=[True, False, False, False], kind="stable"
    ).iloc[0]["Modelo"]
    if not globals().get("conclusiones_generadas", False):
        _reportar_estado("selección de 6.3 deriva de los valores reales", "PENDIENTE", "aún no se ejecutó la rama completa de 6.3")
    else:
        resumen_esperado = resultados_df.set_index("Modelo").loc[seleccion_esperada]
        resumen = globals().get("conclusion_metricas_resumen") or {}
        metricas_coherentes = (globals().get("conclusion_modelo_recomendado") == seleccion_esperada
                               and all(np.isclose(resumen.get(m, np.nan), resumen_esperado[m]) for m in
                                       ("Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"))
                               and int(resumen.get("Falsos_negativos_M", -1)) == conteos_recalculados[seleccion_esperada])
        _reportar_estado("selección y cifras de 6.3 coinciden con métricas/FN", "PASS" if metricas_coherentes else "FAIL")

# Una prueba local no sustituye la ejecución completa desde una sesión limpia de Google Colab.
_reportar_estado("compatibilidad integral y ejecución limpia en Google Colab", "PENDIENTE", "requiere reiniciar y ejecutar todo en Colab con las secciones integradas")

---

### Lista de comprobación de entrega

- [ ] **Marco:** configuración, contrato de variables, cálculo centralizado de métricas, análisis de errores y cierre.
- [ ] **Poma:** carga verificable, limpieza, `X`/`y` correctamente definidos y tabla de resultados.
- [ ] **Aldana:** tres bloques de EDA, regresión logística, curvas ROC y síntesis.
- [ ] **Brickzon:** partición estratificada, escalamiento sin fuga, controles de preparación, KNN y SVM.
- [ ] **Vera:** árbol de decisión, Random Forest, matrices de confusión e importancia de características.
- [ ] **Equipo:** todos los modelos están registrados; se respetó `1 = M`; se compararon los cinco clasificadores sobre el **mismo** conjunto de prueba.
- [ ] **Marco (integración):** ejecución completa en Google Colab desde sesión limpia, sin errores y con conclusiones basadas en métricas reales.

> **Estado inicial de esta plantilla:** todas las celdas de código están deliberadamente vacías. Marcar casillas solamente después de implementar y verificar cada entrega.